In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from skimage.restoration import denoise_nl_means, estimate_sigma

PROJECT_ROOT = Path.cwd().parent
ORIGINAL_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed"
SPLITS = ["train", "validation", "test"]

print("OpenCV:", cv2.__version__)
print()

for split in SPLITS:
    n_images = len(list((ORIGINAL_DIR / split / "images").glob("*.png")))
    n_masks = len(list((ORIGINAL_DIR / split / "masks").glob("*.png")))
    print(f"{split:<11} images: {n_images:>3}   masks: {n_masks:>3}")

sample_path = sorted((ORIGINAL_DIR / "train" / "images").glob("*.png"))[0]
sample_image = Image.open(sample_path)
sample_array = np.array(sample_image)

print()
print("Sample file:", sample_path.name)
print("Mode:", sample_image.mode)
print("Shape:", sample_array.shape)
print("Dtype:", sample_array.dtype)
print("Value range:", sample_array.min(), "to", sample_array.max())

In [ ]:
def gaussian_filter(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)


def median_filter(img, ksize):
    return cv2.medianBlur(img, ksize)


def bilateral_filter(img, d, sigma_color, sigma_space):
    return cv2.bilateralFilter(img, d, sigma_color, sigma_space)


def nlm_filter(img, h):
    return cv2.fastNlMeansDenoising(
        img, None, h=h, templateWindowSize=7, searchWindowSize=21
    )


CANDIDATES = {
    "Gaussian": [
        ("sigma=0.8", lambda im: gaussian_filter(im, 0.8)),
        ("sigma=1.5", lambda im: gaussian_filter(im, 1.5)),
        ("sigma=2.5", lambda im: gaussian_filter(im, 2.5)),
    ],
    "Median": [
        ("k=3", lambda im: median_filter(im, 3)),
        ("k=5", lambda im: median_filter(im, 5)),
        ("k=7", lambda im: median_filter(im, 7)),
    ],
    "Bilateral": [
        ("d=7, sc=25, ss=5", lambda im: bilateral_filter(im, 7, 25, 5)),
        ("d=9, sc=50, ss=7", lambda im: bilateral_filter(im, 9, 50, 7)),
        ("d=9, sc=75, ss=9", lambda im: bilateral_filter(im, 9, 75, 9)),
    ],
    "Non-Local Means": [
        ("h=5", lambda im: nlm_filter(im, 5)),
        ("h=10", lambda im: nlm_filter(im, 10)),
        ("h=15", lambda im: nlm_filter(im, 15)),
    ],
}

# One benign training image, cropped around its lesion for a clear zoom
preview_path = sorted((ORIGINAL_DIR / "train" / "images").glob("benign_*.png"))[0]
preview_mask_path = ORIGINAL_DIR / "train" / "masks" / preview_path.name

preview_img = np.array(Image.open(preview_path))
preview_mask = np.array(Image.open(preview_mask_path)) > 0

ys, xs = np.where(preview_mask)
margin = 40
y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin, 256)
x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin, 256)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for row, (method, settings) in enumerate(CANDIDATES.items()):
    axes[row, 0].imshow(preview_img[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=255)
    axes[row, 0].set_title("Original")
    axes[row, 0].set_ylabel(method, fontsize=12)

    for col, (label, fn) in enumerate(settings, start=1):
        filtered = fn(preview_img)
        axes[row, col].imshow(filtered[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(label)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"Despeckling candidates (training image: {preview_path.name})", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
# Add a milder bilateral setting (built on a copy so re-running is safe)
CANDIDATES_EXT = {m: list(s) for m, s in CANDIDATES.items()}
CANDIDATES_EXT["Bilateral"].insert(
    0, ("d=5, sc=15, ss=3", lambda im: bilateral_filter(im, 5, 15, 3))
)

K3 = np.ones((3, 3), np.uint8)


def load_train_lesions():
    items = []
    for p in sorted((ORIGINAL_DIR / "train" / "images").glob("*.png")):
        if not p.name.startswith(("benign", "malignant")):
            continue
        img = np.array(Image.open(p))
        mask = (np.array(Image.open(ORIGINAL_DIR / "train" / "masks" / p.name)) > 0).astype(np.uint8)
        if mask.sum() == 0:
            continue
        items.append((p.name, img, mask))
    return items


def get_regions(img, mask):
    inner = cv2.erode(mask, K3, iterations=3) > 0
    boundary = (mask - cv2.erode(mask, K3)) > 0
    near = cv2.dilate(mask, K3, iterations=3)
    far = cv2.dilate(mask, K3, iterations=15)
    ring = (far - near) > 0

    # exclude black padding bands (judged on the ORIGINAL image)
    row_ok = img.max(axis=1) > 0
    col_ok = img.max(axis=0) > 0
    ring &= row_ok[:, None] & col_ok[None, :]
    return inner, boundary, ring


def measure(img, inner, boundary, ring):
    f = img.astype(np.float32)
    hf = f - cv2.GaussianBlur(f, (0, 0), 2.0)
    texture = hf[ring].std() / (f[ring].mean() + 1e-6)

    gx = cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=3)
    boundary_grad = np.hypot(gx, gy)[boundary].mean()

    contrast = f[ring].mean() - f[inner].mean()
    return texture, boundary_grad, contrast


train_lesions = load_train_lesions()
print(f"Training lesion images: {len(train_lesions)}")

# Precompute regions and original-image metrics once
prepared = []
for name, img, mask in train_lesions:
    inner, boundary, ring = get_regions(img, mask)
    if inner.sum() < 10 or boundary.sum() == 0 or ring.sum() < 50:
        continue
    prepared.append((name, img, inner, boundary, ring, measure(img, inner, boundary, ring)))

print(f"Images used for metrics: {len(prepared)}")
orig_means = np.mean([p[5] for p in prepared], axis=0)

rows = []
for method, settings in CANDIDATES_EXT.items():
    for label, fn in settings:
        vals = []
        for name, img, inner, boundary, ring, _ in prepared:
            vals.append(measure(fn(img), inner, boundary, ring))
        m = np.mean(vals, axis=0)
        rows.append({
            "method": method,
            "setting": label,
            "texture_ratio": m[0] / orig_means[0],
            "boundary_grad_ratio": m[1] / orig_means[1],
            "contrast_ratio": m[2] / orig_means[2],
        })
    print(f"done: {method}")

comparison_train = pd.DataFrame(rows).round(3)
print()
print(comparison_train.to_string(index=False))

In [ ]:
BOUNDARY_THRESHOLD = 0.80

DESPECKLE_RESULTS_DIR = PROJECT_ROOT / "results" / "despeckling"
DESPECKLE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

comparison_train.to_csv(
    DESPECKLE_RESULTS_DIR / "parameter_screening_train.csv", index=False
)

eligible = comparison_train[
    comparison_train["boundary_grad_ratio"] >= BOUNDARY_THRESHOLD
]

selected = (
    eligible.sort_values("texture_ratio")
    .groupby("method", as_index=False)
    .first()
)
selected = selected.set_index("method").loc[list(CANDIDATES_EXT.keys())].reset_index()

selected.to_csv(DESPECKLE_RESULTS_DIR / "selected_parameters.csv", index=False)

print(
    "Rule: lowest texture ratio with boundary gradient ratio >= "
    f"{BOUNDARY_THRESHOLD}\n"
)
print(selected.to_string(index=False))

print("\nSensitivity to the threshold (method -> chosen setting):")
for thr in [0.70, 0.80, 0.90]:
    el = comparison_train[comparison_train["boundary_grad_ratio"] >= thr]
    pick = el.sort_values("texture_ratio").groupby("method").first()["setting"]
    print(f"  {thr:.2f}: {pick.to_dict()}")

In [ ]:
import shutil
import filecmp

DESPECKLED_ROOT = PROJECT_ROOT / "data" / "processed" / "despeckled"

# Final settings chosen by the training-only rule (selected_parameters.csv)
FINAL_FILTERS = {
    "gaussian": lambda im: gaussian_filter(im, 0.8),
    "median": lambda im: median_filter(im, 3),
    "bilateral": lambda im: bilateral_filter(im, 7, 25, 5),
    "non_local_means": lambda im: nlm_filter(im, 15),
}

print("Selected settings from the screening step:")
print(selected[["method", "setting"]].to_string(index=False))
print()

# --- Generate ---
for method, fn in FINAL_FILTERS.items():
    for split in SPLITS:
        src_img_dir = ORIGINAL_DIR / split / "images"
        src_mask_dir = ORIGINAL_DIR / split / "masks"
        dst_img_dir = DESPECKLED_ROOT / method / split / "images"
        dst_mask_dir = DESPECKLED_ROOT / method / split / "masks"
        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_mask_dir.mkdir(parents=True, exist_ok=True)

        for src in sorted(src_img_dir.glob("*.png")):
            img = np.array(Image.open(src))
            Image.fromarray(fn(img)).save(dst_img_dir / src.name)
            shutil.copy2(src_mask_dir / src.name, dst_mask_dir / src.name)
    print(f"generated: {method}")

# --- Verify by reloading what was saved ---
per_image_rows = []
summary_rows = []

for method in FINAL_FILTERS:
    for split in SPLITS:
        orig_img_dir = ORIGINAL_DIR / split / "images"
        orig_mask_dir = ORIGINAL_DIR / split / "masks"
        new_img_dir = DESPECKLED_ROOT / method / split / "images"
        new_mask_dir = DESPECKLED_ROOT / method / split / "masks"

        orig_names = sorted(p.name for p in orig_img_dir.glob("*.png"))
        new_names = sorted(p.name for p in new_img_dir.glob("*.png"))
        new_mask_names = sorted(p.name for p in new_mask_dir.glob("*.png"))

        masks_identical = (
            new_mask_names == orig_names
            and all(
                filecmp.cmp(orig_mask_dir / n, new_mask_dir / n, shallow=False)
                for n in orig_names
            )
        )

        shape_ok = True
        diffs = []
        for n in orig_names:
            a = np.array(Image.open(orig_img_dir / n))
            b = np.array(Image.open(new_img_dir / n))
            if b.shape != (256, 256) or b.dtype != np.uint8:
                shape_ok = False
            d = float(np.abs(a.astype(np.float32) - b.astype(np.float32)).mean())
            diffs.append(d)
            per_image_rows.append(
                {"method": method, "split": split, "image_name": n, "mean_abs_diff": d}
            )

        summary_rows.append({
            "method": method,
            "split": split,
            "n_images": len(new_names),
            "names_match": new_names == orig_names,
            "masks_identical": masks_identical,
            "shape_dtype_ok": shape_ok,
            "min_mean_abs_diff": round(min(diffs), 3),
            "avg_mean_abs_diff": round(float(np.mean(diffs)), 3),
        })

verification = pd.DataFrame(summary_rows)
print()
print(verification.to_string(index=False))

pd.DataFrame(per_image_rows).to_csv(
    DESPECKLE_RESULTS_DIR / "despeckled_manifest.csv", index=False
)

all_ok = (
    verification["names_match"].all()
    and verification["masks_identical"].all()
    and verification["shape_dtype_ok"].all()
    and (verification["min_mean_abs_diff"] > 0).all()
)
print()
print("ALL CHECKS PASSED" if all_ok else "CHECK FAILED - review the table above")

In [ ]:
spot_path = sorted((ORIGINAL_DIR / "train" / "images").glob("malignant_*.png"))[0]
name = spot_path.name

conditions = {"original": ORIGINAL_DIR, **{m: DESPECKLED_ROOT / m for m in FINAL_FILTERS}}

orig_img = np.array(Image.open(spot_path))
mask = np.array(Image.open(ORIGINAL_DIR / "train" / "masks" / name)) > 0

ys, xs = np.where(mask)
margin = 50
y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin, 256)
x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin, 256)

fig, axes = plt.subplots(2, 5, figsize=(20, 8.5))

for col, (cond, root) in enumerate(conditions.items()):
    img = np.array(Image.open(root / "train" / "images" / name))
    mask_c = np.array(Image.open(root / "train" / "masks" / name)) > 0

    ax = axes[0, col]
    ax.imshow(img[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=255)
    ax.contour(mask_c[y0:y1, x0:x1], levels=[0.5], colors="lime", linewidths=1.2)
    ax.set_title(cond)

    diff = np.abs(img.astype(np.float32) - orig_img.astype(np.float32)) * 5
    axes[1, col].imshow(diff[y0:y1, x0:x1], cmap="magma", vmin=0, vmax=255)
    axes[1, col].set_title("|diff| x5" if col else "(reference)")

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"Spot check across conditions (training image: {name})", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()